# Random Forest Parcel Volume Forecasting

This notebook continues the forecasting project by testing a nonlinear Random Forest model against the existing baseline and Linear Regression results.

> The dataset is synthetic and contains no confidential DHL data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATA_PATH = Path("../data/dhl_style_parcel_volume.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

## Recreate the forecasting features

The model uses recent parcel demand and calendar information. The data remains sorted by depot and date so lag features represent the correct historical observations.

In [ ]:
df = df.sort_values(["depot", "date"]).copy()

df["lag_1"] = df.groupby("depot")["parcel_volume"].shift(1)
df["lag_7"] = df.groupby("depot")["parcel_volume"].shift(7)
df["month"] = df["date"].dt.month
df["day_of_week_num"] = df["date"].dt.dayofweek

model_df = df.dropna(subset=["lag_1", "lag_7"]).copy()

features = [
    "lag_1",
    "lag_7",
    "day_of_week_num",
    "month",
    "is_bank_holiday",
    "is_black_friday_period",
    "is_christmas_peak",
]

## Time-based train/test split

As before, observations before 1 January 2026 are used for training and 2026 observations are held out for testing.

In [ ]:
split_date = "2026-01-01"

train_df = model_df[model_df["date"] < split_date].copy()
test_df = model_df[model_df["date"] >= split_date].copy()

X_train = train_df[features]
y_train = train_df["parcel_volume"]
X_test = test_df[features]
y_test = test_df["parcel_volume"]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

## Baseline and Linear Regression references

In [ ]:
baseline_mae = mean_absolute_error(test_df["parcel_volume"], test_df["lag_7"])

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)
linear_mae = mean_absolute_error(y_test, linear_predictions)

## Train Random Forest

Random Forest combines predictions from many decision trees, allowing it to model nonlinear relationships and interactions between features.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_predictions)

results = pd.DataFrame({
    "Model": ["7-day lag baseline", "Linear Regression", "Random Forest"],
    "Test MAE": [baseline_mae, linear_mae, rf_mae],
})

results

In [ ]:
rf_improvement = ((baseline_mae - rf_mae) / baseline_mae) * 100
rf_vs_linear = ((linear_mae - rf_mae) / linear_mae) * 100

print(f"Random Forest MAE: {rf_mae:.2f}")
print(f"Improvement vs baseline: {rf_improvement:.1f}%")
print(f"Improvement vs Linear Regression: {rf_vs_linear:.1f}%")

## Feature importance

Random Forest exposes impurity-based feature importances. These help describe which inputs the fitted model used most strongly, but they should not be interpreted as causal effects.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_,
}).sort_values(by="Importance", ascending=False)

feature_importance

In [ ]:
feature_importance.sort_values("Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    legend=False,
    figsize=(9, 5),
)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Interpretation

With the current dataset, `lag_7` is the dominant feature at roughly **74.7%** importance. This indicates that parcel demand on the same day one week earlier is the strongest predictor in this fitted Random Forest. Bank holidays, day of week and yesterday's volume provide additional adjustments.

### Current result

- Baseline MAE: approximately **1,182.84**
- Linear Regression MAE: approximately **1,029.12**
- Random Forest MAE: approximately **903.34**
- Random Forest improvement vs baseline: approximately **23.6%**

The next modelling stage will test Gradient Boosting and richer time-series features.